This is a p good model for maybe the top end analysis. Seems to be doing pretty well

In [2]:
from transformers import pipeline

# Load the classification pipeline with the specified model
pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis", return_all_scores=False)

# Classify a new sentence
good_sentence = "I think shawn is a great guy"
bad_sentnece = "abood has been really shitty to me lately"
sentence = 'I think Shawn is a great guy but Abood has been really shitty to me lately'
result = pipe(bad_sentnece)

# Print the result
print(result)

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\pipelines\text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


[{'label': 'Negative', 'score': 0.5759602189064026}]


Interest model: https://huggingface.co/firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3

Defo needs more testing, but interesting. It generalizes the whole entry tho. This will need some way to parse sentences which is weird. Maybe this can be used as an over-the-top model? idk

In [2]:
# Requires: librosa
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
import librosa
import torch
import numpy as np

model_id = "firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3"
model = AutoModelForAudioClassification.from_pretrained(model_id)

feature_extractor = AutoFeatureExtractor.from_pretrained(model_id, do_normalize=True)
id2label = model.config.id2label

def preprocess_audio(audio_path, feature_extractor, max_duration=30.0):
    audio_array, sampling_rate = librosa.load(audio_path, sr=None)
    
    max_length = int(feature_extractor.sampling_rate * max_duration)
    if len(audio_array) > max_length:
        audio_array = audio_array[:max_length]
    else:
        audio_array = np.pad(audio_array, (0, max_length - len(audio_array)))

    inputs = feature_extractor(
        audio_array,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=max_length,
        truncation=True,
        return_tensors="pt",
    )
    return inputs

def predict_emotion(audio_path, model, feature_extractor, id2label, max_duration=30.0):
    inputs = preprocess_audio(audio_path, feature_extractor, max_duration)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_id = torch.argmax(logits, dim=-1).item()
    predicted_label = id2label[predicted_id]
    
    return predicted_label

audio_path = r"C:\Users\User\projects\MindLine\testing_models\test_audios\Test3.wav"

predicted_emotion = predict_emotion(audio_path, model, feature_extractor, id2label)
print(f"Predicted Emotion: {predicted_emotion}")


Predicted Emotion: angry


Sarcasm detector cuz sarcasm is still huge

This one below is kinda mid

In [3]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
import string

def preprocess_data(text: str) -> str:
   return text.lower().translate(str.maketrans("", "", string.punctuation)).strip()

MODEL_PATH = "helinivan/english-sarcasm-detector"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

text = "Yeah, I love the pain"
tokenized_text = tokenizer([preprocess_data(text)], padding=True, truncation=True, max_length=256, return_tensors="pt")
output = model(**tokenized_text)
probs = output.logits.softmax(dim=-1).tolist()[0]
confidence = max(probs)
prediction = probs.index(confidence)
results = {"is_sarcastic": prediction, "confidence": confidence}
print(results)


KeyboardInterrupt: 

I think we found the one for emotions gang LESGO

In [4]:
from transformers import pipeline

classifier = pipeline(task="text-classification", model="SamLowe/roberta-base-go_emotions", top_k=None)

sentences = [bad_sentnece] 

model_outputs = classifier(sentences)
model_outputs[0]
# produces a list of dicts for each of the labels


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[{'label': 'annoyance', 'score': 0.4621132016181946},
 {'label': 'disappointment', 'score': 0.3438815474510193},
 {'label': 'disgust', 'score': 0.12094248831272125},
 {'label': 'anger', 'score': 0.08203191310167313},
 {'label': 'disapproval', 'score': 0.06776023656129837},
 {'label': 'neutral', 'score': 0.05365602672100067},
 {'label': 'sadness', 'score': 0.04677429050207138},
 {'label': 'approval', 'score': 0.009553383104503155},
 {'label': 'embarrassment', 'score': 0.00941687636077404},
 {'label': 'realization', 'score': 0.008362304419279099},
 {'label': 'desire', 'score': 0.0036978404968976974},
 {'label': 'optimism', 'score': 0.0035749583039432764},
 {'label': 'fear', 'score': 0.0029588225297629833},
 {'label': 'admiration', 'score': 0.0024782842956483364},
 {'label': 'love', 'score': 0.0023896032944321632},
 {'label': 'gratitude', 'score': 0.001833559712395072},
 {'label': 'caring', 'score': 0.001797381672076881},
 {'label': 'nervousness', 'score': 0.001745060901157558},
 {'label'

Testing out the entity detection options

In [3]:
import spacy

sentences = [
    'Today was a great day, man—like I answered really well in all my classes and even got all my Calculus-II homework done on time.', 
    'On top of that, I hit it off with this guy named Shawn and had a lot of fun.', 
    'I think Shawn is a great dude, and Shawn and I would make amazing friends—like, how can this day get any better?'
]

nlp = spacy.load("en_core_web_sm")

for text in sentences:
    print('----')
    doc = nlp(text)
    
    # Iterate through every individual word/token
    for token in doc:
        if token.pos_ in ["PROPN", "NOUN"]:
            print(f"{token.text: <15} | Tag: {token.pos_}")

----
Today           | Tag: NOUN
day             | Tag: NOUN
man             | Tag: NOUN
classes         | Tag: NOUN
Calculus        | Tag: PROPN
II              | Tag: PROPN
homework        | Tag: NOUN
time            | Tag: NOUN
----
top             | Tag: NOUN
guy             | Tag: NOUN
Shawn           | Tag: PROPN
lot             | Tag: NOUN
fun             | Tag: NOUN
----
Shawn           | Tag: PROPN
dude            | Tag: NOUN
Shawn           | Tag: PROPN
friends         | Tag: NOUN
day             | Tag: NOUN


In [260]:
import spacy

nlp = spacy.load("en_core_web_sm")

sentences = [
    # "I think Shawn is a great, smart and intelligent guy",
    # 'I think Shawn is a great guy but Abood has been really shitty to me lately'
    # 'Shawn is my best friend'
    # 'Shawn, my best friend from childhood, is incredibly talented and helpful, but Abood is a person who is often rude and nasty.'
    # 'The tall, ancient trees and bright, colorful flowers filled the quiet garden.'
    "Suraj, Jacob and Abood are all dumb"
]

for text in sentences:
    print(f"\n--- Sentence Analysis ---")
    doc = nlp(text)
    
    print(f"{'Token':<12} | {'Universal POS':<15} | {'Fine-Grained Tag'}")
    print("-" * 50)
    
    for token in doc:
        print(f"{token.text:<12} | {token.pos_:<15} | {token.dep_}")


--- Sentence Analysis ---
Token        | Universal POS   | Fine-Grained Tag
--------------------------------------------------
Suraj        | PROPN           | nsubj
,            | PUNCT           | punct
Jacob        | PROPN           | conj
and          | CCONJ           | cc
Abood        | PROPN           | conj
are          | AUX             | ROOT
all          | ADV             | dep
dumb         | ADJ             | acomp


## Nouns and Adjectives

In [259]:
import spacy

nlp = spacy.load("en_core_web_sm")
sentence = 'The tall, ancient trees and bright, colorful flowers filled the quiet garden.'
sentence2 = 'She wore a long, flowing dress and shiny, silver shoes to the elegant evening gala.'
sentence3 = 'Shawn, my best friend from childhood, is incredibly talented and helpful, but Abood is a person who is often rude and nasty.'
sentence4 = 'The brilliant, innovative ideas of Albert Einstein and the meticulous, groundbreaking research of Marie Curie changed modern science.'
sentence5 = 'The brilliant Albert Einstein developed revolutionary theories about space and time.'
sentence6 = 'The legendary Michael Jackson transformed the global music industry.'
sentence7 = 'Suraj, Jacob and Abood are all dumb'
doc = nlp(sentence7)

def find_target_noun(adj_token):
    # Candidate list to store (noun_token, priority_score)
    candidates = []

    # Case 1: The adjective modifies a noun directly (amod)
    if adj_token.dep_ == "amod":
        candidates.append(adj_token.head)

    # Case 2: The adjective is a complement (e.g., "Shawn is great")
    # We look at the head (usually a verb) and find its subject
    elif adj_token.dep_ == "acomp" or adj_token.head.pos_ == "AUX":
        for child in adj_token.head.children:
            if child.dep_ in ["nsubj", "attr"]:
                candidates.append(child)

    if not candidates:
        return None

    return candidates[0]

# Execution
for token in doc:
    if token.pos_ == 'ADJ':
        target = find_target_noun(token)
        target_text = target.text if target else "Unknown"
        print(f"Adjective: {token.text:10} | Linked to: {target_text} ({target.pos_ if target else 'N/A'})")

Adjective: dumb       | Linked to: Suraj (PROPN)


In [88]:
import spacy

nlp = spacy.load("en_core_web_sm")

sentence = 'I think Shawn is a great guy, but Abood has been really shitty to me lately'
sentencef2 = "Shawn isn't actually mean. Shawn is actually kinda nice" 
sentencef = 'Abood is a man who is bitter.' 
sentence3 = 'Shawn, my best friend from childhood, is incredibly talented and helpful, but Abood is a person who is often rude and nasty.'
sentence4 = "Shawn and Abood are so smart"
conj_test_sentence = "I think Shawn is a great, smart and intelligent guy"
verb_sentence = 'I like Shawn'
sentence5 = "Cats are amazing, Dogs are dirty and stinky"
sentence6 = 'The brilliant Albert Einstein developed revolutionary theories about space and time.'
sentence7= 'The legendary Michael Jackson transformed the global music industry.'
sentence8 = 'The brilliant, innovative ideas of Albert Einstein and the meticulous, groundbreaking research of Marie Curie changed modern science.'
sentence9 = 'Fiona wore a long, flowing dress and shiny, silver shoes to the elegant evening gala.'

holy_test = 'The tall, ancient trees and bright, colorful flowers filled the quiet garden.'

# Testing multiple nouns/propns
test1 = "Suraj, Jacob and Abood are all dumb"
test2 = "Jacob is just as dumb as Caleb"
test3 = "Cats and dogs are both stupid and dumb"
test4 = "Abood and Ajeet are awful."
test5 = "Shawn and Abood are smart, sexy and awesome"
test6 = "Shawn is smart, sexy and awesome and so is Abood"

adjectives = []

def get_doc_stats(doc):
    res = {}
    for token in doc:
        if token.pos_ not in res:
            res[token.pos_] = [token.text]
        
        else:
            res[token.pos_].append(token.text)
    
    return res

def find_noun_adj_pairs(doc, token, token_counter):
    candidates = []
    adjs = []
    start_token_number = token_counter
    end_token_number = token_counter
    curr_token = token

    if token.dep_ == "amod":
        candidates.append(token.head)
    
    elif token.dep_ == "acomp" or token.head.pos_ == "AUX":
        for child in token.head.children:
            if child.dep_ in ["nsubj", "attr"]:
                candidates.append(child)

    while curr_token.pos_ not in ['PUNCT', 'CCONJ'] and start_token_number > 0:
        start_token_number -= 1
        curr_token = doc[start_token_number]

    sub_token_range = doc[start_token_number:end_token_number]
    for sub_token in sub_token_range:
        if sub_token.pos_ in ["PROPN", "NOUN"]:
            candidates.append(sub_token)
            # adjectives.append((token.text, sub_token.text))
    
    # candidates.sort(key=lambda x: x.pos_ != "PROPN")

    conseq_nouns = find_conseq_nouns(doc)
    if candidates:
        if len(conseq_nouns) > 0:
            for n in conseq_nouns:
                candidates.append(n)
        for propn in candidates:
            adjs.append((propn.text, token.text))
        
    else:
        adjs = []

    return adjs

def consolidate_conj(doc, token, token_counter):
    candidates = []
    ads = []
    adjs = []
    start_token_number = token_counter
    end_token_number = token_counter
    curr_token = token

    ads.append(curr_token)

    # Traversing back thru the sentence, picking up adjectives until
    # noun/propn is found
    while token.text != '.' and start_token_number > 0:
        start_token_number -= 1
        curr_token = doc[start_token_number]

        if curr_token.pos_ in ["PROPN"]:
            break
    
    sub_token_range = doc[start_token_number:end_token_number]
    # print(sub_token_range)
    for sub_token in sub_token_range:
        if sub_token.pos_ in ["NOUN", "PROPN"]:
            candidates.append(sub_token)

        if sub_token.dep_ in ["amod", "acomp"]:
            ads.append(sub_token)
        
    # candidates.sort(key=lambda x: x.pos_ != "PROPN")
    # if candidates:
        # propn = candidates
    # else:
        # adjs = []
    
    conseq_nouns = find_conseq_nouns(doc)
    if len(conseq_nouns) > 0:
            for n in conseq_nouns:
                candidates.append(n)

    for noun in candidates:
        for adj in ads:
            adjs.append((noun.text, adj.text))
    
    return adjs

def find_conseq_nouns(doc):
    index = 0
    desired = ['PROPN', 'NOUN']
    conseq_nouns = []

    token = doc[0]
    while index < (len(doc) - 2) and token.nbor(1):
        if token.pos_ in desired:
            neighbors = [token.nbor(1), token.nbor(2)]
            if neighbors[0].pos_ in desired or neighbors[1].pos_ in desired:
                conseq_nouns.append(token)
                conseq_nouns.append(neighbors[0]) if neighbors[0].pos_ in desired else conseq_nouns.append(neighbors[1])
                token = token.nbor(2)
                index += 2
            else:
                token = token.nbor(1)
                index += 1 
        else:
            token.nbor(1)
            index += 1

    return list(set(conseq_nouns))
    
def handle_proper_nouns(sentence):
    doc = nlp(sentence)
    token_counter = 0
    for token in doc:
        # print(token.text, '-', token.pos_)
        if token.pos_ in ['ADJ']:
            if token.dep_ in ["amod", "acomp"]:
                pairs = find_noun_adj_pairs(doc, token, token_counter)
                for i in pairs:
                    adjectives.append(i)
            
            elif token.dep_ in ['conj']:
                pairs = consolidate_conj(doc, token, token_counter)
                for i in pairs:
                    adjectives.append(i)
                
        token_counter += 1

    final = set(adjectives)

    return final

pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis", return_all_scores=False)

results = []
# doc = nlp(test2)
# # print(check_conseq_propns(doc))


print(test4)
proper_nouns = handle_proper_nouns(test4)
proper_nouns
# doc = nlp(test1)
# print(get_doc_stats(doc))
# print(proper_nouns)

# doc = nlp(test5)
# find_conseq_nouns(doc)
# for ad in final:
    # result = pipe(ad[0])
#     print(ad[1], '-', ad[0], '-', result)
    

Abood and Ajeet are awful.


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\pipelines\text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


{('Abood', 'awful'), ('Ajeet', 'awful')}